<img src="https://hvplot.holoviz.org/_static/logo_horizontal.svg" width="20%" align="right"/>

# Big data visualization with Dask and hvPlot

In this notebook, we'll learn to use the hvPlot APIs with Dask DataFrames.

---

## Reconnect to our Dask Cluster

In [1]:
!pip install dask-gateway


In [2]:
import dask_gateway
import dask.dataframe as dd

In [3]:
from dask.distributed import Client, LocalCluster

# Khởi tạo cụm quản lý cục bộ trên Colab
cluster = LocalCluster()

# Kết nối client điều khiển
client = Client(cluster)

# Hiển thị thông tin quản lý và Dask Dashboard
cluster

INFO:distributed.http.proxy:To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
INFO:distributed.scheduler:State start
INFO:distributed.diskutils:Found stale lock file and directory '/tmp/dask-scratch-space/scheduler-puidi4zk', purging
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:37475
INFO:distributed.scheduler:  dashboard at:  http://127.0.0.1:8787/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:34777'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:37879'
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:41475 name: 0
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:41475
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:58074
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:37867 name: 1
INFO:distributed.scheduler:Sta

LocalCluster(0107d27b, 'tcp://127.0.0.1:37475', workers=2, threads=2, memory=12.67 GiB)

## Load a subset of columns

In [4]:
columns = [
    'YEAR', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK', 'FL_DATE', 'OP_CARRIER',
    'TAIL_NUM', 'OP_CARRIER_FL_NUM', 'ORIGIN', 'DEST', 'CRS_DEP_TIME',
    'DEP_TIME', 'DEP_DELAY', 'ARR_TIME', 'ARR_DELAY', 'CANCELLED',
    'CANCELLATION_CODE', 'DIVERTED', 'AIR_TIME', 'FLIGHTS', 'DISTANCE',
    'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY',
    'LATE_AIRCRAFT_DELAY', 'DIV_ARR_DELAY'
]

In [5]:
import dask.dataframe as dd
import requests
import zipfile
import io
import os

# 1. Tải dữ liệu tháng 1/2022
url = (
    "https://transtats.bts.gov/PREZIP/"
    "On_Time_Reporting_Carrier_On_Time_Performance_"
    "1987_present_2022_1.zip"
)

r = requests.get(url)
r.raise_for_status()

# 2. Giải nén CSV vào /content
with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    csv_file = [f for f in z.namelist() if f.endswith(".csv")][0]
    z.extract(csv_file, "/content")

csv_path = os.path.join("/content", csv_file)

print("Đã tải file:")
print(csv_path)

# 3. Các cột dùng trong bài 06
columns = [
    'Year', 'Month', 'DayofMonth', 'DayOfWeek',
    'FlightDate', 'Reporting_Airline',
    'Tail_Number', 'Flight_Number_Reporting_Airline',
    'Origin', 'Dest', 'CRSDepTime',
    'DepTime', 'DepDelay',
    'ArrTime', 'ArrDelay',
    'Cancelled', 'CancellationCode',
    'Diverted', 'AirTime',
    'Flights', 'Distance',
    'CarrierDelay', 'WeatherDelay',
    'NASDelay', 'SecurityDelay',
    'LateAircraftDelay'
]

# 4. Đọc bằng Dask
flights = dd.read_csv(
    csv_path,
    usecols=lambda col: col in columns,
    dtype={
        'CancellationCode': 'object',
        'FlightDate': 'string',
        'ArrTime': 'float64',
        'ArrDelay': 'float64',
        'DepTime': 'float64',
        'DepDelay': 'float64',
        'Distance': 'float64'
    },
    assume_missing=True
)

# 5. Xem 5 dòng đầu
flights.head()

Đã tải file:
/content/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2022_1.csv


,Year,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Tail_Number,Flight_Number_Reporting_Airline,Origin,Dest,...,CancellationCode,Diverted,AirTime,Flights,Distance,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,2022.0,1.0,14.0,5.0,2022-01-14,YX,N119HQ,4879.0,CMH,DCA,...,<NA>,0.0,63.0,1.0,323.0,NaN,NaN,NaN,NaN,NaN
1,2022.0,1.0,15.0,6.0,2022-01-15,YX,N122HQ,4879.0,CMH,DCA,...,<NA>,0.0,50.0,1.0,323.0,NaN,NaN,NaN,NaN,NaN
2,2022.0,1.0,16.0,7.0,2022-01-16,YX,N412YX,4879.0,CMH,DCA,...,<NA>,0.0,53.0,1.0,323.0,NaN,NaN,NaN,NaN,NaN
3,2022.0,1.0,17.0,1.0,2022-01-17,YX,N405YX,4879.0,CMH,DCA,...,<NA>,0.0,56.0,1.0,323.0,NaN,NaN,NaN,NaN,NaN
4,2022.0,1.0,18.0,2.0,2022-01-18,YX,N420YX,4879.0,CMH,DCA,...,<NA>,0.0,48.0,1.0,323.0,NaN,NaN,NaN,NaN,NaN


In [6]:
flights.head()

,Year,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Tail_Number,Flight_Number_Reporting_Airline,Origin,Dest,...,CancellationCode,Diverted,AirTime,Flights,Distance,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,2022.0,1.0,14.0,5.0,2022-01-14,YX,N119HQ,4879.0,CMH,DCA,...,<NA>,0.0,63.0,1.0,323.0,NaN,NaN,NaN,NaN,NaN
1,2022.0,1.0,15.0,6.0,2022-01-15,YX,N122HQ,4879.0,CMH,DCA,...,<NA>,0.0,50.0,1.0,323.0,NaN,NaN,NaN,NaN,NaN
2,2022.0,1.0,16.0,7.0,2022-01-16,YX,N412YX,4879.0,CMH,DCA,...,<NA>,0.0,53.0,1.0,323.0,NaN,NaN,NaN,NaN,NaN
3,2022.0,1.0,17.0,1.0,2022-01-17,YX,N405YX,4879.0,CMH,DCA,...,<NA>,0.0,56.0,1.0,323.0,NaN,NaN,NaN,NaN,NaN
4,2022.0,1.0,18.0,2.0,2022-01-18,YX,N420YX,4879.0,CMH,DCA,...,<NA>,0.0,48.0,1.0,323.0,NaN,NaN,NaN,NaN,NaN


## hvPlot + Dask

To use hvPlot's build in Dask integration, we need to switch out:

`import hvplot.pandas` for `import hvplot.dask`

In [7]:
!pip install jupyter_bokeh datashader hvplot

In [8]:
!pip install hvplot

In [9]:
!pip install bokeh

In [10]:
import hvplot.dask
hvplot.extension('bokeh')

### Plot the departure delay per day for the entire dataset

In [11]:
flights.groupby('FlightDate')['DepDelay'].count().hvplot()

:Curve   [FlightDate]   (DepDelay)

### 💻 Your turn: Visualize the weekly distribution of the mean of any variable in the datasets

You can any plot type from the [hvPlot Gallery](https://hvplot.holoviz.org/reference/index.html)

In [12]:
# Your code here. When ready, click on the three dots below for the solution.
flights.groupby('DayOfWeek')['ArrDelay'].mean().hvplot.scatter(
    x='DayOfWeek',
    y='ArrDelay'
)

:Scatter   [DayOfWeek]   (ArrDelay)

In [13]:
flights.groupby('DayOfWeek')['ArrDelay'].mean().hvplot.scatter(x="DayOfWeek", y='ArrDelay')

:Scatter   [DayOfWeek]   (ArrDelay)

## Plotting large datasets

In the above visualization of daily counts we saw a bunch of compute happening before we saw the plot appear. But after it was generated, panning and zooming did not cause any new Dask computes.

This is because the final dataset after the groupby is only about `20 years * 365 days` long, so it fits completely in memory.

Now let's look at the entire dataset:

In [14]:
print(f"The full dataset has {len(flights)/1e6:2} million rows")

The full dataset has 0.537902 million rows


If we try and send these many data points to the browser for visualization in a plot, the browser would run out of memory and crash.

<img src="images/datashader.svg" width="30%" align="right">

The solution for this is to take advantage of the fact that the output plot has a fixed resolution in terms of number of pixels. A 600x400 image has 240,000 pixels. This means that if we plotted 125 million points on the these pixels, most would overlay each other and not be visible. Instead, we pre-render or rasterize the data and shade in a manner that maintains an accurate the distribution of your data.

We do this via the hvPlot integration with **Datashader**.

We will use a smaller dataset for the next few examples for quick outputs. These examples will work with the full dataset, but will take a bit longer to run with the 10 computer nodes we are currently using for this tutorial.

In [15]:
import dask.dataframe as dd

# Đọc file CSV bằng các tham số chuẩn của Dask CSV
flights = dd.read_csv(
    "/content/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2022_1*",
    usecols=lambda col: col in columns, # Lọc cột
    dtype={'CancellationCode': 'object'},
    assume_missing=True
)

# Nếu muốn lọc năm > 2017, hãy dùng cú pháp lọc trực tiếp của DataFrame
flights = flights[flights['Year'] > 2017]

In [16]:
flights.head()

,Year,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Tail_Number,Flight_Number_Reporting_Airline,Origin,Dest,...,CancellationCode,Diverted,AirTime,Flights,Distance,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,2022.0,1.0,14.0,5.0,2022-01-14,YX,N119HQ,4879.0,CMH,DCA,...,<NA>,0.0,63.0,1.0,323.0,NaN,NaN,NaN,NaN,NaN
1,2022.0,1.0,15.0,6.0,2022-01-15,YX,N122HQ,4879.0,CMH,DCA,...,<NA>,0.0,50.0,1.0,323.0,NaN,NaN,NaN,NaN,NaN
2,2022.0,1.0,16.0,7.0,2022-01-16,YX,N412YX,4879.0,CMH,DCA,...,<NA>,0.0,53.0,1.0,323.0,NaN,NaN,NaN,NaN,NaN
3,2022.0,1.0,17.0,1.0,2022-01-17,YX,N405YX,4879.0,CMH,DCA,...,<NA>,0.0,56.0,1.0,323.0,NaN,NaN,NaN,NaN,NaN
4,2022.0,1.0,18.0,2.0,2022-01-18,YX,N420YX,4879.0,CMH,DCA,...,<NA>,0.0,48.0,1.0,323.0,NaN,NaN,NaN,NaN,NaN


In [17]:
print(f"The smaller dataset has {len(flights)/1e6} million rows")

The smaller dataset has 0.537902 million rows


In these next two visualizations, Datashader data is displayed on the plots.
As we pan and zoom, Datashader recomputes the appropriate pixel shades using Dask.

This allows us to easily look at the entire 30 million row dataset, but still
zoom into a single point, without requiring downsampling or decimation of the dataset.

In [18]:
!pip install datashader

In [19]:
import hvplot.dask
import panel as pn
pn.extension()

In [20]:
flights['FlightDate'] = dd.to_datetime(flights['FlightDate'])
flights.hvplot.line(x='FlightDate', y='DepDelay', datashade=True)

BokehModel(combine_events=True, render_bundle={'docs_json': {'f14ed8f0-bdae-4303-b2e4-61e760bee94d': {'version…

In [21]:
import pandas as pd
import hvplot.pandas

# Gom nhóm tính trung bình DepDelay theo FlightDate và chuyển về Pandas
daily_df = flights.groupby('FlightDate')['DepDelay'].mean().compute().reset_index()
daily_df['FlightDate'] = pd.to_datetime(daily_df['FlightDate'])

# Vẽ biểu đồ đường
daily_df.hvplot.line(x='FlightDate', y='DepDelay', title="Trung bình phút trễ theo ngày")

:Curve   [FlightDate]   (DepDelay)

In [22]:
flights[['ArrDelay', 'Distance']].hvplot.scatter(x='ArrDelay', y='Distance', datashade=True)

BokehModel(combine_events=True, render_bundle={'docs_json': {'b6b323dc-d60b-4a06-a885-4895df1b9d04': {'version…

In [ ]:
# To shutdown the cluster, uncomment and run the next line
# cluster.shutdown()

---

## Next →

[Big data dashboards](./07-big-data-dashboards.ipynb)